# EDA — PUBG Finish Placement Prediction

> Разведочный анализ для проекта *PUBG Placement Predictor + Drift Monitoring*.

## Оглавление

1. [Постановка задачи](#s1)
2. [Настройки окружения](#s2)
3. [Обзор значений и пропусков](#s3)
4. [Режимы матчей](#s4)
5. [Утечка killPlace](#s5)
6. [Схема признаков: IN / OUT](#s6)
7. [Сетка таргета](#s7)
8. [Baseline](#s8)

In [1]:
import pandas as pd
import numpy  as np
from pathlib  import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics         import mean_absolute_error
import lightgbm as lgb

<a id="s1"></a>
## 1. Постановка задачи

**Задача.** Регрессия: предсказать `winPlacePerc` [0, 1] — нормированное финальное
место игрока в матче (1 — победа, 0 — последнее место). Метрика - MAE.

**Данные.** Kaggle *PUBG Finish Placement Prediction*, `train_V2.csv`.
Одна строка = агрегированная статистика одного игрока за один матч.

<a id="s2"></a>
## 2. Настройки окружения

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in {"research", "notebooks"} else Path.cwd()
RAW_DIR      = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH   = RAW_DIR / "train_V2.csv"

<a id="s3"></a>
## 3. Обзор значений,пропусков, аномалий

In [3]:
df = pd.read_csv(TRAIN_PATH)
df.head()

,Id,groupId,matchId,assists,boosts,damageDealt,DBNOs,headshotKills,heals,killPlace,...,revives,rideDistance,roadKills,swimDistance,teamKills,vehicleDestroys,walkDistance,weaponsAcquired,winPoints,winPlacePerc
0,7f96b2f878858a,4d4b580de459be,a10357fd1a4a91,0,0,0.00,0,0,0,60,...,0,0.0000,0,0.00,0,0,244.80,1,1466,0.4444
1,eef90569b9d03c,684d5656442f9e,aeb375fc57110c,0,0,91.47,0,0,0,57,...,0,0.0045,0,11.04,0,0,1434.00,5,0,0.6400
2,1eaf90ac73de72,6a4a42c3245a74,110163d8bb94ae,1,0,68.00,0,0,0,47,...,0,0.0000,0,0.00,0,0,161.80,2,0,0.7755
3,4616d365dd2853,a930a9c79cd721,f1f1f4ef412d7e,0,0,32.90,0,0,0,75,...,0,0.0000,0,0.00,0,0,202.70,3,0,0.1667
4,315c96c26c9aac,de04010b3458dd,6dc8ff871e21e6,0,0,100.00,0,0,0,45,...,0,0.0000,0,0.00,0,0,49.75,2,0,0.1875


In [4]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 4446966 entries, 0 to 4446965
Data columns (total 29 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   Id               4446966 non-null  str    
 1   groupId          4446966 non-null  str    
 2   matchId          4446966 non-null  str    
 3   assists          4446966 non-null  int64  
 4   boosts           4446966 non-null  int64  
 5   damageDealt      4446966 non-null  float64
 6   DBNOs            4446966 non-null  int64  
 7   headshotKills    4446966 non-null  int64  
 8   heals            4446966 non-null  int64  
 9   killPlace        4446966 non-null  int64  
 10  killPoints       4446966 non-null  int64  
 11  kills            4446966 non-null  int64  
 12  killStreaks      4446966 non-null  int64  
 13  longestKill      4446966 non-null  float64
 14  matchDuration    4446966 non-null  int64  
 15  matchType        4446966 non-null  str    
 16  maxPlace         4446966 non-

**1 пропуск `winPlacePerc`**

In [5]:
df = df.dropna(subset=["winPlacePerc"])

### Фильтр читеров
аномальные значения, которые выявляют подозрительных игроков

In [6]:
total_dist = df['walkDistance'] + df['rideDistance'] + df['swimDistance']

anomalies = {
    "Убийства без перемещения":     (df['kills'] > 0) & (total_dist == 0),
    "100% headshot при kills > 10":  (df['kills'] >= 10) & (df['headshotKills'] == df['kills']),
    "Убийства с расстояния > 1км":  (df['longestKill'] > 1000),
    "Подобрано оружия >50":         (df['weaponsAcquired'] > 50),
    "Аномально много убийств":      (df['kills'] > 40)
}

report = pd.DataFrame({
    "строк": {k: int(m.sum())           for k, m in anomalies.items()},
    "доля%": {k: round(100*m.mean(), 4) for k, m in anomalies.items()}
})
print(report.sort_values("строк", ascending=False))
print(f'Итоговая доля подозрительных игроков {round(report["доля%"].sum(),2)}%')


                              строк   доля%
Убийства без перемещения       1535  0.0345
Подобрано оружия >50            165  0.0037
Аномально много убийств          33  0.0007
100% headshot при kills > 10     24  0.0005
Убийства с расстояния > 1км      21  0.0005
Итоговая доля подозрительных игроков 0.04%


<a id="s4"></a>
## 4. Режимы матчей

Live-данные - ranked squad fpp на больших картах, сопоставимый режим в датасете - squad-fpp, обе выборки фильтруются до него

In [7]:
df["matchType"].value_counts()

matchType
squad-fpp           1756186
duo-fpp              996691
squad                626526
solo-fpp             536761
duo                  313591
solo                 181943
normal-squad-fpp      17174
crashfpp               6287
normal-duo-fpp         5489
flaretpp               2505
normal-solo-fpp        1682
flarefpp                718
normal-squad            516
crashtpp                371
normal-solo             326
normal-duo              199
Name: count, dtype: int64

In [8]:
df = df[df["matchType"] == 'squad-fpp']
df.shape


(1756186, 29)

<a id="s5"></a>
## 5. Утечка `killPlace`
`killPlace` - ранг по количеству убийств, при равном - тай брейк по порядку выбывания -> кодирует таргет. док-во: 

In [9]:
zero_kills = df[df['kills'] == 0]
corr_all = df['killPlace'].corr(df['winPlacePerc'])
corr_zero = zero_kills['killPlace'].corr(zero_kills['winPlacePerc'])
print(f'все игроки:  {corr_all}')
print(f'с 0 убийств: {corr_zero}')


все игроки:  -0.6962698860204445
с 0 убийств: -0.9388873053018071


при нуле убийств почти идеальная корреляция с `killPlace`

Вывод: `killPlace` исключаем из фич, как и `rankPoints`, `killPoints`, `winPoints` - в современном api их нет

In [10]:
df = df.drop(['killPlace', 'rankPoints', 'killPoints', 'winPoints'], axis = 1)
df.shape


(1756186, 25)

<a id="s6"></a>
## 6. Схема признаков: IN / OUT

После фильтра режима и удаления утечки/легаси остаётся 20 фич модели (маппятся 1:1 Kaggle ↔ PUBG API).

**IN — 20 фич:**

| Группа | Фичи |
|---|---|
| Бой | `damageDealt`, `kills`, `killStreaks`, `headshotKills`, `DBNOs`, `longestKill`, `assists`, `roadKills`, `teamKills` |
| Поддержка / выживание | `heals`, `boosts`, `revives` |
| Перемещение | `walkDistance`, `rideDistance`, `swimDistance`, `vehicleDestroys`, `weaponsAcquired` |
| Структура матча | `maxPlace`, `numGroups`, `matchDuration` |

**OUT:**

| Колонка(и) | Причина |
|---|---|
| `winPlacePerc` | таргет |
| `killPlace` | утечка — при `kills==0` corr ≈ −0.90 (тай-брейк кодирует исход) |
| `rankPoints`, `killPoints`, `winPoints` | легаси — нет в современном API, в Kaggle половина `−1/0` |
| `Id` | идентификатор строки, не признак |
| `matchId`, `groupId` | служебные — сплит по матчам |
| `matchType` | фильтр режима (после фильтра — константа) |

`maxPlace`/`numGroups` оставлены: структура матча, вычислимы из ростера; `maxPlace` нужен для снапа предсказаний к сетке таргета.

<a id="s7"></a>
## 7. Сетка таргета

`winPlacePerc = (maxPlace - winPlace) / (maxPlace - 1)` - таргет квантован шагом `1/(maxPlace - 1)`.
Предсказание снапим к этой сетке на постпроцессинге

In [11]:
# Берём один матч
match  = df[df["matchId"] == df["matchId"].iloc[0]]
m      = match["maxPlace"].iloc[0]
places = np.sort(match["winPlacePerc"].unique())

# Индексы должны быть целыми (в пределах погрешности)
k = places * (m - 1)

print("maxPlace:", m)
print(np.round(k, 6))


maxPlace: 28
[ 0.      0.999   2.0007  2.9997  3.9987  5.0004  5.9994  7.0011  8.0001
  8.9991 10.0008 10.9998 11.9988 13.0005 13.9995 15.0012 16.0002 16.9992
 19.9989 21.0006 21.9996 23.0013 24.0003 24.9993 26.001  27.    ]


подтверждено

<a id="s8"></a>
## 8. Baseline
Сплит строго по `matchId`, строки внутри одного матча взаимосвязаны

In [12]:
TARGET   = "winPlacePerc"
DROP     = [TARGET, "Id", "groupId", "matchId", "matchType"]
FEATURES = [c for c in df.columns if c not in DROP]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=1)
tr_idx, va_idx = next(splitter.split(df, groups=df["matchId"]))
train, val     = df.iloc[tr_idx], df.iloc[va_idx]


### Среднее

In [13]:
mean_pred = np.full(len(val), train[TARGET].mean())
mae_mean  = mean_absolute_error(val[TARGET], mean_pred)


### LightGBM

In [16]:
model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=1)
model.fit(train[FEATURES], train[TARGET])
lgb_pred = np.clip(model.predict(val[FEATURES]), 0.0, 1.0)
mae_lgb  = mean_absolute_error(val[TARGET], lgb_pred)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024954 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1760
[LightGBM] [Info] Number of data points in the train set: 1404222, number of used features: 20
[LightGBM] [Info] Start training from score 0.464783


In [17]:
print(f'MAE среднего {mae_mean} \nMAE LightGBM {mae_lgb}')

MAE среднего 0.2720346198772089 
MAE LightGBM 0.09808615895858268


# TODO

- Графики фич
- аггрегация по groupid
- обоснование численных решений в отлове аномалий (через boxplot ?)
- feature importance достать из LGBM
- baseline среднего по выборке плох, распределение равномерное, и так понятен результат
- фолды вместо groupshufflesplit
- анализ пропуска в таргете который тупа выкинул